# 82 Boxes: Data Exploration

Exploring the 2025-26 regular season from the NBA API. Findings are recorded in `docs/data_notes.md`. 

Pipeline: `src/extraact_data.py` saves the raw pull to `data/raw/`, and `src/clean_data.py` builds the final tables in `data/processed/`.This notebook is the exploration record: it shows the API calls and how each decision was reached, while never writing data files. Pulling the shot charts is too slow to repeat here (about 3 minutes), so section 5 loads the file saved by `extract_data.py`


## 0. Setup


In [20]:
import pandas as pd
from nba_api.stats.endpoints import leaguegamefinder, playergamelogs, shotchartdetail

# Ensuring we can see every available column
pd.set_option("display.max_columns", None)

## 1. Player game log
    
    
  One row per player per game they appeared in 

In [ ]:
players = playergamelogs.PlayerGameLogs(
    season_nullable="2025-26",
    season_type_nullable="Regular Season",
    timeout=60,
).get_data_frames()[0]

players.shape # 26651 rows x 72 columns
players.head(10)




,SEASON_YEAR,PLAYER_ID,PLAYER_NAME,NICKNAME,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,TOV,STL,BLK,BLKA,PF,PFD,PTS,PLUS_MINUS,NBA_FANTASY_PTS,DD2,TD3,WNBA_FANTASY_PTS,FP_HIGH_SCORE,GP_RANK,W_RANK,L_RANK,W_PCT_RANK,MIN_RANK,FGM_RANK,FGA_RANK,FG_PCT_RANK,FG3M_RANK,FG3A_RANK,FG3_PCT_RANK,FTM_RANK,FTA_RANK,FT_PCT_RANK,OREB_RANK,DREB_RANK,REB_RANK,AST_RANK,TOV_RANK,STL_RANK,BLK_RANK,BLKA_RANK,PF_RANK,PFD_RANK,PTS_RANK,PLUS_MINUS_RANK,NBA_FANTASY_PTS_RANK,DD2_RANK,TD3_RANK,WNBA_FANTASY_PTS_RANK,FP_HIGH_SCORE_RANK,AVAILABLE_FLAG,MIN_SEC,TEAM_COUNT
0,2025-26,1642948,Ryan Nembhard,Ryan,1610612742,DAL,Dallas Mavericks,0022501193,2026-04-12T00:00:00,DAL vs. CHI,W,38.283333,6,11,0.545,1,2,0.500,2,2,1.000,2,7,9,23,4,2,0,1,2,3,15,23,62.3,1,0,52.0,76.0,1,1,1,1,902,5150,7089,8084,8796,14467,4020,6628,8167,1,3403,1633,1893,1,857,1684,8231,17813,12346,4569,6828,857,216,1,137,606,65,1,38:17,1
1,2025-26,1642852,Derik Queen,Derik,1610612740,NOP,New Orleans Pelicans,0022501195,2026-04-12T00:00:00,NOP @ MIN,L,41.053333,11,28,0.393,1,2,0.500,7,8,0.875,7,15,22,4,4,1,1,5,5,5,30,-5,64.4,1,0,61.0,66.0,1,13431,13431,13431,234,629,59,16436,8796,14467,4020,912,1073,7125,79,20,3,4684,857,5231,2514,26621,25321,1653,782,17487,153,1,137,175,240,1,41:03,1
2,2025-26,1630567,Scottie Barnes,Scottie,1610612761,TOR,Toronto Raptors,0022501192,2026-04-12T00:00:00,TOR vs. BKN,W,31.991667,8,11,0.727,0,0,0.000,2,2,1.000,0,12,12,12,1,1,3,0,0,4,18,35,61.4,1,1,50.0,66.0,1,1,1,1,5153,2433,7089,2779,15203,21411,15203,6628,8167,1,14357,124,601,139,9238,5231,270,1,1,2814,4719,78,241,1,1,802,240,1,31:60,1
3,2025-26,1642847,Jeremiah Fears,Jeremiah,1610612740,NOP,New Orleans Pelicans,0022501195,2026-04-12T00:00:00,NOP @ MIN,L,41.383333,12,28,0.429,0,5,0.000,12,14,0.857,3,7,10,5,2,2,1,3,3,8,36,0,62.5,1,0,57.0,65.0,1,13431,13431,13431,209,355,59,14210,15203,6031,15203,98,103,7305,1661,1633,1348,3163,4472,1684,2514,26113,18467,283,235,12504,209,1,137,304,271,1,41:23,1
4,2025-26,1642866,Joan Beringer,Joan,1610612750,MIN,Minnesota Timberwolves,0022501195,2026-04-12T00:00:00,MIN vs. NOP,W,31.266667,9,12,0.750,0,0,0.000,6,9,0.667,6,7,13,2,0,0,7,1,3,6,24,-1,63.6,1,0,53.0,62.0,1,1,1,1,5818,1615,5869,2168,15203,21411,15203,1394,785,9873,180,1633,417,10190,16881,13156,2,17813,18467,889,2070,13633,168,1,137,534,398,1,31:16,1
5,2025-26,1631248,Baylor Scheierman,Baylor,1610612738,BOS,Boston Celtics,0022501186,2026-04-12T00:00:00,BOS vs. ORL,W,38.806667,8,20,0.400,6,14,0.429,8,8,1.000,0,7,7,7,3,2,1,0,3,7,30,15,54.9,0,0,56.0,60.0,1,1,1,1,725,2433,910,15410,169,83,7542,588,1073,1,14357,1633,3776,1397,2001,1684,2514,1,18467,513,782,2649,562,1917,137,345,488,1,38:48,1
6,2025-26,1642382,Branden Carlson,Branden,1610612760,OKC,Oklahoma City Thunder,0022501196,2026-04-12T00:00:00,OKC vs. PHX,L,42.233333,10,20,0.500,5,10,0.500,1,2,0.500,1,9,10,1,1,2,5,4,0,2,26,-18,59.5,1,0,56.0,59.0,1,13431,13431,13431,152,997,910,8975,420,682,4020,10847,8167,11175,7058,610,1348,14618,9238,1684,21,26529,1,7467,1506,24826,317,1,137,345,544,1,42:14,1
7,2025-26,1631123,Jamaree Bouyea,Jamaree,1610612756,PHX,Phoenix Suns,0022501196,2026-04-12T00:00:00,PHX @ OKC,W,39.683333,12,23,0.522,3,8,0.375,0,2,0.000,0,5,5,9,4,1,2,1,1,2,27,37,51.5,0,0,50.0,59.0,1,1,1,1,453,355,398,8934,2309,1811,8971,14120,8167,14120,14357,4120,7318,593,857,5231,779,17813,5688,7467,1290,49,856,1917,137,802,544,1,39:41,1
8,2025-26,1628389,Bam Adebayo,Bam,1610612748,MIA,Miami Heat,0022501189,2026-04-12T00:00:00,MIA vs. ATL,W,29.233333,11,18,0.611,1,4,0.250,2,4,0.500,2,8,10,3,0,3,2,0,0,3,25,16,56.5,1,0,49.0,56.0,1,1,1,1,7689,629,1556,5689,8796,8381,12158,6628,4158,11175,3403,998,1348,6970,16881,466,779,1,1,4569,1777,2336,476,1,137,921,771,1,29:14,1
9,2025-26,1630194,Paul Reed,Paul,1610612765,DET,Detroit Pistons,0022501188,2026-04-12T00:00:00,DET @ IND,W,21.716667,11,11,1.000,0,0,0.000,4,4,1.000,2,4,6,3,0,3,3,0,1,3,26,28,55.7,0,0,47.0,56.0,1,1,1,1,14825,629,7089,1,15203

## 2. Team game log 

One row per team per game. 30 teams x 82 games = 2460 rows (expected)

In [34]:
teams = leaguegamefinder.LeagueGameFinder(
    season_nullable="2025-26",
    season_type_nullable="Regular Season",
    timeout=60,
).get_data_frames()[0]

teams.shape # 2460 rows x 28 columns (82 games x 30 teams = 2460 games)
teams.head(10)





,SEASON_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,PTS,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,STL,BLK,TOV,PF,PLUS_MINUS
0,22025,1610612752,NYK,New York Knicks,0022501190,2026-04-12,NYK vs. CHA,L,240,96,39,90,0.433,13,40,0.325,5,8,0.625,12,27,39,25,10,3,10,25,-14.0
1,22025,1610612762,UTA,Utah Jazz,0022501198,2026-04-12,UTA @ LAL,L,240,107,41,94,0.436,7,34,0.206,18,27,0.667,17,25,42,25,13,4,16,14,-24.0
2,22025,1610612754,IND,Indiana Pacers,0022501188,2026-04-12,IND vs. DET,L,239,121,41,90,0.456,23,54,0.426,16,20,0.800,8,23,31,31,15,6,16,17,-12.0
3,22025,1610612756,PHX,Phoenix Suns,0022501196,2026-04-12,PHX @ OKC,W,241,135,56,101,0.554,20,41,0.488,3,6,0.500,13,41,54,28,5,8,11,9,32.0
4,22025,1610612745,HOU,Houston Rockets,0022501194,2026-04-12,HOU vs. MEM,W,240,132,48,105,0.457,14,49,0.286,22,30,0.733,21,43,64,23,10,8,11,10,31.0
5,22025,1610612760,OKC,Oklahoma City Thunder,0022501196,2026-04-12,OKC vs. PHX,L,240,103,41,95,0.432,18,46,0.391,3,9,0.333,9,27,36,32,9,7,10,8,-32.0
6,22025,1610612753,ORL,Orlando Magic,0022501186,2026-04-12,ORL @ BOS,L,241,108,36,91,0.396,12,43,0.279,24,30,0.800,14,36,50,22,6,5,19,19,-5.0
7,22025,1610612738,BOS,Boston Celtics,0022501186,2026-04-12,BOS vs. ORL,W,241,113,36,87,0.414,19,50,0.380,22,22,1.000,8,38,46,24,10,6,17,23,5.0
8,22025,1610612750,MIN,Minnesota Timberwolves,0022501195,2026-04-12,MIN vs. NOP,W,239,132,45,89,0.506,10,37,0.270,32,43,0.744,12,35,47,32,5,13,11,27,6.0
9,22025,1610612741,CHI,Chicago Bulls,0022501193,2026-04-12,CHI @ DAL,L,241,128,52,106,0.491,10,38,0.263,14,19,0.737,13,29,42,30,12,3,13,20,-21.0


### 2a. Home and away teams 

`MATCHUP` is `"BKN vs. IND"` for a Brooklyn home game and `"BKN @ TOR"` for a Brooklyn away game. 

In [ ]:
teams["is_home"] = teams["MATCHUP"].str.contains(" vs. ", regex = False)

teams["home_team"] = teams["TEAM_ABBREVIATION"].where(
    teams["is_home"], teams["MATCHUP"].str.split(" @ ").str[1]
)
teams["away_team"] = teams["MATCHUP"].str.split(" vs. ").str[1].where(
    teams["is_home"], teams["MATCHUP"].str.split(" @ ").str[0]
)

# Expecting exactly one home row per game (1230 total unique regular season games)
teams.groupby("GAME_ID")["is_home"].sum().value_counts()
# 1 = 1225, 0 = 5. We have 5 games that have no "vs." row. 



is_home
1    1225
0       5
Name: count, dtype: int64

### 2b. Finding the 5 games with no home row

These five games are returning is_home as false since they were played at neutral-sites. For example the NYK vs ORL and SAS vs OKC games were all played in Vegas for the NBA cup. The other game were played at international courts.

In [37]:
teams[teams.groupby("GAME_ID")["is_home"].transform("sum") == 0]


,SEASON_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,PTS,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,STL,BLK,TOV,PF,PLUS_MINUS,is_home,home_team,away_team
1201,22025,1610612763,MEM,Memphis Grizzlies,0022500602,2026-01-18,ORL @ MEM,W,240,126,48,91,0.527,13,35,0.371,17,25,0.680,9,33,42,31,12,7,12,18,17.0,False,MEM,ORL
1207,22025,1610612753,ORL,Orlando Magic,0022500602,2026-01-18,ORL @ MEM,L,239,109,42,95,0.442,10,33,0.303,15,16,0.938,12,30,42,30,5,6,17,20,-17.0,False,MEM,ORL
1243,22025,1610612763,MEM,Memphis Grizzlies,0022500578,2026-01-15,MEM @ ORL,L,240,111,38,83,0.458,17,36,0.472,18,21,0.857,7,30,37,27,5,6,16,18,-7.0,False,ORL,MEM
1253,22025,1610612753,ORL,Orlando Magic,0022500578,2026-01-15,MEM @ ORL,W,240,118,41,95,0.432,15,40,0.375,21,23,0.913,19,35,54,28,9,4,16,15,7.0,False,ORL,MEM
1712,22025,1610612759,SAS,San Antonio Spurs,0022501230,2025-12-13,SAS @ OKC,W,241,111,37,90,0.411,13,42,0.310,24,32,0.750,15,37,52,21,9,6,19,23,2.0,False,OKC,SAS
1713,22025,1610612753,ORL,Orlando Magic,0022501229,2025-12-13,NYK @ ORL,L,241,120,45,97,0.464,13,42,0.310,17,25,0.680,12,29,41,28,9,3,14,24,-12.0,False,ORL,NYK
1714,22025,1610612752,NYK,New York Knicks,0022501229,2025-12-13,NYK @ ORL,W,240,132,51,84,0.607,7,20,0.350,23,31,0.742,8,33,41,28,9,7,12,23,12.0,False,ORL,NYK
1715,22025,1610612760,OKC,Oklahoma City Thunder,0022501230,2025-12-13,SAS @ OKC,L,240,109,38,92,0.413,9,37,0.243,24,27,0.889,14,37,51,20,13,4,14,25,-2.0,False,OKC,SAS
2294,22025,1610612765,DET,Detroit Pistons,0022500147,2025-11-01,DAL @ DET,W,239,122,45,91,0.495,9,24,0.375,23,34,0.676,14,33,47,29,10,4,7,22,12.0,False,DET,DAL
2295,22025,1610612742,DAL,Dallas Mavericks,0022500147,2025-11-01,DAL @ DET,L,241,110,34,80,0.425,16,45,0.356,26,31,0.839,7,36,43,23,3,3,15,25,-12.0,False,DET,DAL


## 3. Games table

One row per game(1230). Cleaning adds `season`, `game_type`, `neutral_site` and team IDs. 

In [23]:
# Making games dataframe
games = teams[["GAME_ID","GAME_DATE","home_team","away_team"]].drop_duplicates("GAME_ID")

print(games.shape) # 1230 rows x 4 columns 
# 1230 unique games in a season


(1230, 4)


## 4. Can team stats be derived from player stats? 

Sum the 14 counting stats over each team's player rows and compare with the team log. 

In [ ]:
sum_cols = ["PTS", "FGM", "FGA", "FG3M", "FG3A", "FTM", "FTA",
            "OREB", "DREB", "AST", "STL", "BLK", "TOV", "PF"]


# Sum player rows per (game, team)
from_players = players.groupby(["GAME_ID", "TEAM_ID"])[sum_cols].sum()

# Team log stats, rows lined up with above table
from_team = teams.set_index(["GAME_ID", "TEAM_ID"])[sum_cols].reindex(from_players.index)

print("Team-games compared:", len(from_players)) # expecting 2460


diff = (from_players != from_team).any(axis=1)
print("Team-games that differ:", diff.sum()) # 3 




Team-games compared: 2460
Team-games that differ: 3


PTS  FGM  FGA  FG3M  FG3A  FTM  FTA  OREB  DREB  AST  \
GAME_ID    TEAM_ID                                                            
0022500243 1610612755    0    0    0     0     0    0    0     0    -1    0   
0022500792 1610612745    0    0    1     0     0    0    0     0     0    0   
           1610612766    0    0    0     0     0    0    0     0     0    0   

                       STL  BLK  TOV  PF  
GAME_ID    TEAM_ID                        
0022500243 1610612755    0    0    0   0  
0022500792 1610612745    0    0    0   0  
           1610612766    0    1    0   0

In [ ]:
(from_players - from_team)[diff]
# 3 instances where the data does not align: 
# GAME_ID         TEAM_ID        REASON 
# 0022500243	1610612755	    Defensive Rebounds is off by (-1)
# 0022500792	1610612745      FGA is off by (+1)
# 0022500792	1610612766	    BLK is off by (+1)

## 5. Shot chart
Loaded from `data/raw/shot_chart.parquet`. The pull itself (one API call per team, about 3 minutes) lives in `src/extract_data.py`.

Note: a single league-wide `ShotChartDetail` call is capped at 102,400 rows (573 games), which is why the pull loops over teams.

In [38]:
shot_chart = pd.read_parquet("../data/raw/shot_chart.parquet")

print(shot_chart.shape)  # 219160 rows x 24 columns
print("Games:", shot_chart["GAME_ID"].nunique())  # 1230
print("Total FGA in player log:", players["FGA"].sum())  # 219160, equal to the row count


(219160, 24)
Games: 1230
Total FGA in player log: 219160


### 5a. Trim the columns we keep 

In [ ]:
shots = shot_chart.drop(columns=["GRID_TYPE", "PLAYER_NAME", "TEAM_NAME",
                                 "SHOT_ATTEMPTED_FLAG", "GAME_DATE"])
print(shots.shape) # 219160 rows x 19 columns 

(219160, 19)


### 5b. Is (GAME_ID, GAME_EVENT_ID) a valid primary key? 

In [ ]:
print("Duplicate (GAME_ID, GAME_EVENT_ID):", shots.duplicated(["GAME_ID", "GAME_EVENT_ID"]).sum())
# There are no duplicate pairs of GAME_ID, EVENT_ID, so that pair will work as our primary key for the shots table. 

Duplicate (GAME_ID, GAME_EVENT_ID): 0


### 5c. Cross-check: home team parsed from matchup vs HTM in the shot data

In [28]:
check = games.merge(shots[["GAME_ID", "HTM"]].drop_duplicates(), on="GAME_ID")
print("Home team mismatches:", (check["home_team"] != check["HTM"]).sum())
# Result: 0 mismatches across all 1,230 games, including the 5 neutral-site games.

Home team mismatches: 0


In [29]:
# Finding player id for Egor Demin (Brooklyn Nets)
players[players["PLAYER_NAME"].str.contains("Egor")][["PLAYER_ID","PLAYER_NAME","TEAM_ID"]].drop_duplicates()
#PLAYER_ID	PLAYER_NAME	TEAM_ID
#1642856	Egor Dëmin	1610612751

,PLAYER_ID,PLAYER_NAME,TEAM_ID
7438,1642856,Egor Dëmin,1610612751
